# Notebook 05: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

In this notebook, we use **LoRA (Low-Rank Adaptation)** to fine-tune the Moirai encoder.
Instead of updating millions of parameters, LoRA freezes the foundation model and injects tiny trainable rank-decomposition matrices into the linear layers.

uv pip install peft

In [1]:
import torch
import pandas as pd
from IPython.display import display

from heads import MeanPoolingClassifier, SingleScaleAttentionClassifier, SingleScaleMultiHeadClassifier, HierarchicalMultiHeadClassifier
from models import LoraHeadWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [1e-4]
wd_grid = [0.05, 0.01]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from utils import set_seed
set_seed(42)
print("Seed 42 — experiments are reproducible.")

Seed 42 — experiments are reproducible.


In [3]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [4]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 2. LoRA Baseline: Mean Pooling on Full Sequence

In [5]:
methods = ["LoRA", "DoRA", "AdaLoRA"]
df_mean_pool = pd.DataFrame(index=methods, columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"
df_mean_pool.index.name = "PEFT Method"

for method in methods:
    print(f"\n{'='*40}\nTraining with {method}\n{'='*40}")
    for patch in PATCH_SIZES:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_mean_pool.loc[method, patch] = metrics["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MeanPool-{method}"] = metrics

print("Mean Pooling (LoRA/DoRA/AdaLoRA) - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))


Training with LoRA
LR=0.0001| WD=0.05


 Early stopping : 39
Val Loss: 1.0496
LR=0.0001| WD=0.01
 Early stopping : 38
Val Loss: 1.0642
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.0894
LR=0.0001| WD=0.01
 Early stopping : 36
Val Loss: 1.0821
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.1349
LR=0.0001| WD=0.01
 Early stopping : 38
Val Loss: 1.0571
LR=0.0001| WD=0.05
 Early stopping : 32
Val Loss: 1.1360
LR=0.0001| WD=0.01
 Early stopping : 30
Val Loss: 1.1279

Training with DoRA
LR=0.0001| WD=0.05
 Early stopping : 37
Val Loss: 1.0954
LR=0.0001| WD=0.01
 Early stopping : 33
Val Loss: 1.0998
LR=0.0001| WD=0.05
 Early stopping : 34
Val Loss: 1.0712
LR=0.0001| WD=0.01
 Early stopping : 30
Val Loss: 1.0923
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.1246
LR=0.0001| WD=0.01
 Early stopping : 35
Val Loss: 1.1071
LR=0.0001| WD=0.05
 Early stopping : 34
Val Loss: 1.1378
LR=0.0001| WD=0.01
 Early stopping : 31
Val Loss: 1.1410

Training with AdaLoRA
LR=0.0001| WD=0.05
 Early stopping : 95
Val Loss: 1.0552
LR=0.00

Patch Size,8,16,32,64
PEFT Method,,,,
LoRA,0.6492,0.6367,0.6346,0.6379
DoRA,0.6326,0.6383,0.6338,0.6350
AdaLoRA,0.6342,0.6415,0.6517,0.6330



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6492,0.4944,0.3853,0.3938,0.6021,0.6492,0.6021
MeanPool-DoRA,0.6326,0.4517,0.3825,0.3896,0.5815,0.6326,0.5947
MeanPool-AdaLoRA,0.6342,0.4478,0.3925,0.4015,0.5800,0.6342,0.5950


## 3. LoRA: Single-Scale Attention

Single learned Query vector that attends over all temporal patches, with LoRA rank=8.
- **shared_context**: one Query across all variables simultaneously
- **independent_context**: one dedicated Query per variable

In [6]:
MODES = ["shared_context", "independent_context"]

df_attn_lora = pd.DataFrame(index=MODES, columns=PATCH_SIZES)
df_attn_lora.index.name = "Mode"
df_attn_lora.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    for mode in MODES:
        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8,
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_attn_lora.loc[mode, patch] = metrics["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode}) LoRA r=8"] = metrics

print("Single-Scale Attention with LoRA r=8 — Accuracy")
display(df_attn_lora.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.1378
LR=0.0001| WD=0.01
 Early stopping : 33
Val Loss: 1.1276
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.1204
LR=0.0001| WD=0.01
 Early stopping : 31
Val Loss: 1.1344
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.0937
LR=0.0001| WD=0.01
 Early stopping : 30
Val Loss: 1.1737
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.1365
LR=0.0001| WD=0.01
 Early stopping : 32
Val Loss: 1.1181
LR=0.0001| WD=0.05
 Early stopping : 34
Val Loss: 1.1526
LR=0.0001| WD=0.01
 Early stopping : 30
Val Loss: 1.1936
LR=0.0001| WD=0.05
 Early stopping : 32
Val Loss: 1.1905
LR=0.0001| WD=0.01
 Early stopping : 31
Val Loss: 1.1516
LR=0.0001| WD=0.05
 Early stopping : 30
Val Loss: 1.1533
LR=0.0001| WD=0.01
 Early stopping : 27
Val Loss: 1.1585
LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.2088
LR=0.0001| WD=0.01
 Early stopping : 29
Val Loss: 1.2011
Single-Scale Attention with LoRA r=8 — Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6148,0.6196,0.6034,0.6180
independent_context,0.6152,0.6285,0.6062,0.6208



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6492,0.4944,0.3853,0.3938,0.6021,0.6492,0.6021
MeanPool-DoRA,0.6326,0.4517,0.3825,0.3896,0.5815,0.6326,0.5947
MeanPool-AdaLoRA,0.6342,0.4478,0.3925,0.4015,0.5800,0.6342,0.5950
Attention (shared_context) LoRA r=8,0.6148,0.3746,0.3440,0.3467,0.5490,0.6148,0.5671
Attention (independent_context) LoRA r=8,0.6152,0.3711,0.3572,0.3538,0.5419,0.6152,0.5679


## 4. LoRA Rank Ablation Study
Testing the impact of the bottleneck rank $r$ on the MHA performance. A higher rank means more trainable parameters.

In [ ]:
PATCH = 8
RANKS_TO_TEST = [2, 4, 8, 16, 32, 64]
methods = ["LoRA", "DoRA", "AdaLoRA"]

df_rank_ablation8 = pd.DataFrame(index=methods, columns=RANKS_TO_TEST)
df_rank_ablation8.index.name = "PEFT Method"
df_rank_ablation8.columns.name = f"Rank 'r' (Patch {PATCH})"

for method in methods:
    for r in RANKS_TO_TEST:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": r,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            lr_grid=lr_grid, wd_grid=wd_grid
        )
        df_rank_ablation8.loc[method, r] = metrics["Accuracy"]
        df_patch_8_metrics.loc[f"{method} r={r}"] = metrics

print(f"Rank Ablation (Patch = {PATCH}) - Accuracy")
display(df_rank_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05


 Early stopping : 53
Val Loss: 1.0724
LR=0.0001| WD=0.01
 Early stopping : 58
Val Loss: 1.0617
LR=0.0001| WD=0.05
 Early stopping : 44
Val Loss: 1.0754
LR=0.0001| WD=0.01
 Early stopping : 54
Val Loss: 1.0363
LR=0.0001| WD=0.05
 Early stopping : 36
Val Loss: 1.1005
LR=0.0001| WD=0.01
 Early stopping : 36
Val Loss: 1.0788
LR=0.0001| WD=0.05
 Early stopping : 35
Val Loss: 1.0738
LR=0.0001| WD=0.01
 Early stopping : 31
Val Loss: 1.0640
LR=0.0001| WD=0.05
 Early stopping : 23
Val Loss: 1.1273
LR=0.0001| WD=0.01
 Early stopping : 28
Val Loss: 1.0761
LR=0.0001| WD=0.05
 Early stopping : 17
Val Loss: 1.1504
LR=0.0001| WD=0.01
 Early stopping : 22
Val Loss: 1.0735
LR=0.0001| WD=0.05
 Early stopping : 51
Val Loss: 1.0946
LR=0.0001| WD=0.01
 Early stopping : 56
Val Loss: 1.0711
LR=0.0001| WD=0.05


## 5. Advanced LoRA: Multi-Head Attention

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 64

model_names_single = [f"MHA (Heads={NUM_HEADS} | LoRA r=8)"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_mha = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = metrics_mha["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MHA-64 ({mode}) LoRA r=8"] = metrics_mha

LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.0864
LR=0.0001| WD=0.01
 Early stopping : 28
Val Loss: 1.1096
LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.0798
LR=0.0001| WD=0.01
 Early stopping : 27
Val Loss: 1.0717
LR=0.0001| WD=0.05
 Early stopping : 25
Val Loss: 1.0335
LR=0.0001| WD=0.01
 Early stopping : 24
Val Loss: 1.0628
LR=0.0001| WD=0.05
 Early stopping : 25
Val Loss: 1.0392
LR=0.0001| WD=0.01
 Early stopping : 25
Val Loss: 1.0230
LR=0.0001| WD=0.05
 Early stopping : 22
Val Loss: 1.1337
LR=0.0001| WD=0.01
 Early stopping : 22
Val Loss: 1.0977
LR=0.0001| WD=0.05
 Early stopping : 23
Val Loss: 1.1231
LR=0.0001| WD=0.01
 Early stopping : 23
Val Loss: 1.1208
LR=0.0001| WD=0.05
 Early stopping : 21
Val Loss: 1.1307
LR=0.0001| WD=0.01
 Early stopping : 20
Val Loss: 1.1475
LR=0.0001| WD=0.05
 Early stopping : 21
Val Loss: 1.1324
LR=0.0001| WD=0.01
 Early stopping : 21
Val Loss: 1.1825


In [ ]:
print("MHA with LoRA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

MHA with LoRA - Accuracy


Patch Size                                          8       16      32      64
Model                      Mode                                               
MHA (Heads=64 | LoRA r=64) shared_context       0.6180  0.6188  0.6188  0.6221
                           independent_context  0.6241  0.6139  0.6127  0.6156


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6492,0.4944,0.3853,0.3938,0.6021,0.6492,0.6021
MeanPool-DoRA,0.6326,0.4517,0.3825,0.3896,0.5815,0.6326,0.5947
MeanPool-AdaLoRA,0.6342,0.4478,0.3925,0.4015,0.5800,0.6342,0.5950
LoRA r=2,0.6464,0.5323,0.3995,0.4177,0.5980,0.6464,0.6062
LoRA r=4,0.6484,0.4745,0.3887,0.3942,0.5977,0.6484,0.6020
LoRA r=8,0.6521,0.4575,0.3891,0.3971,0.5983,0.6521,0.6068
LoRA r=16,0.6371,0.4339,0.3946,0.3962,0.5815,0.6371,0.5983
LoRA r=32,0.6419,0.4304,0.3867,0.3883,0.5844,0.6419,0.6006
LoRA r=64,0.6415,0.4698,0.3805,0.3842,0.5850,0.6415,0.5947


## 5b. MHA Head Count Ablation with LoRA (patch = 8)

Testing `num_heads` ∈ {4, 8, 16, 32, 64, 128, 384} with LoRA rank ∈ {8, 64}, both context modes, patch = 8.

In [ ]:
HEADS_TO_TEST_LORA = [4, 8, 16, 32, 64, 128, 384]
RANKS_MHA          = [8, 64]
MODES_MHA          = ["shared_context", "independent_context"]

df_mha_lora_heads = pd.DataFrame(
    index=pd.MultiIndex.from_product([HEADS_TO_TEST_LORA, MODES_MHA], names=["Num Heads", "Mode"]),
    columns=RANKS_MHA,
)
df_mha_lora_heads.columns.name = "LoRA Rank"

for r in RANKS_MHA:
    for heads in HEADS_TO_TEST_LORA:
        for mode in MODES_MHA:
            print(f"heads={heads:3d}  mode={mode}  r={r}")
            _, metrics = universal_grid_search(
                model_class=LoraHeadWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {
                        "num_vars": NUM_VARS,
                        "num_classes": num_classes,
                        "patch_mode": mode,
                        "num_heads": heads,
                    },
                    "patch_size": 8,
                    "num_vars": NUM_VARS,
                    "size": SIZE,
                    "remove_last_patch": False,
                    "lora_r": r,
                },
                train_loader=tr_loader,
                val_loader=va_loader,
                test_loader=te_loader,
                device=DEVICE,
                wd_grid=wd_grid,
                lr_grid=lr_grid,
            )
            df_mha_lora_heads.loc[(heads, mode), r] = metrics["Accuracy"]
            df_patch_8_metrics.loc[f"MHA-{heads} ({mode}) LoRA r={r}"] = metrics

print("\nMHA LoRA Head Ablation — Accuracy (patch=8)")
display(df_mha_lora_heads.astype(float).round(4))

## 6. Hierarchical MHA (LoRA)

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 64

df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode}) LoRA r=8"] = metrics_hier

print("Hierarchical MHA with LoRA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05
 Early stopping : 30
Val Loss: 1.2527
LR=0.0001| WD=0.01
 Early stopping : 26
Val Loss: 1.2730
LR=0.0001| WD=0.05
 Early stopping : 31
Val Loss: 1.2343
LR=0.0001| WD=0.01
 Early stopping : 26
Val Loss: 1.2640
LR=0.0001| WD=0.05
 Early stopping : 26
Val Loss: 1.2289
LR=0.0001| WD=0.01
 Early stopping : 26
Val Loss: 1.2715
LR=0.0001| WD=0.05
 Early stopping : 23
Val Loss: 1.2432
LR=0.0001| WD=0.01
 Early stopping : 27
Val Loss: 1.2306
LR=0.0001| WD=0.05
 Early stopping : 22
Val Loss: 1.2904
LR=0.0001| WD=0.01
 Early stopping : 24
Val Loss: 1.2760
LR=0.0001| WD=0.05
 Early stopping : 26
Val Loss: 1.2595
LR=0.0001| WD=0.01
 Early stopping : 25
Val Loss: 1.2751
LR=0.0001| WD=0.05
 Early stopping : 24
Val Loss: 1.2768
LR=0.0001| WD=0.01
 Early stopping : 26
Val Loss: 1.2571
LR=0.0001| WD=0.05
 Early stopping : 22
Val Loss: 1.2630
LR=0.0001| WD=0.01
 Early stopping : 22
Val Loss: 1.2864
Hierarchical MHA with LoRA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5823,0.5921,0.5791,0.5819
independent_context,0.5876,0.5929,0.5831,0.5827



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6492,0.4944,0.3853,0.3938,0.6021,0.6492,0.6021
MeanPool-DoRA,0.6326,0.4517,0.3825,0.3896,0.5815,0.6326,0.5947
MeanPool-AdaLoRA,0.6342,0.4478,0.3925,0.4015,0.5800,0.6342,0.5950
LoRA r=2,0.6464,0.5323,0.3995,0.4177,0.5980,0.6464,0.6062
LoRA r=4,0.6484,0.4745,0.3887,0.3942,0.5977,0.6484,0.6020
LoRA r=8,0.6521,0.4575,0.3891,0.3971,0.5983,0.6521,0.6068
LoRA r=16,0.6371,0.4339,0.3946,0.3962,0.5815,0.6371,0.5983
LoRA r=32,0.6419,0.4304,0.3867,0.3883,0.5844,0.6419,0.6006
LoRA r=64,0.6415,0.4698,0.3805,0.3842,0.5850,0.6415,0.5947


In [ ]:
# ── Export Results to CSV ─────────────────────────────────────────────────────
import os
import numpy as np

os.makedirs("results_csv", exist_ok=True)

FINETUNING = "05_LoRA"
MODE_LABEL = {'shared_context': 'shared', 'independent_context': 'indep'}
rows = []

for patch in PATCH_SIZES:
    # Mean Pooling — best PEFT method (no mode)
    best_acc  = float(df_mean_pool[patch].astype(float).max())
    best_peft = df_mean_pool[patch].astype(float).idxmax()
    if patch == 8:
        rec = float(df_patch_8_metrics.loc[f'MeanPool-{best_peft}', 'Macro Recall'])
        f1  = float(df_patch_8_metrics.loc[f'MeanPool-{best_peft}', 'Macro F1'])
    else:
        rec, f1 = float('nan'), float('nan')
    rows.append({'finetuning': FINETUNING, 'pooling': 'Mean', 'patch_size': patch,
                 'accuracy': best_acc, 'macro_recall': rec, 'macro_f1': f1})

    # Single-Scale Attention with LoRA r=8 — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_attn_lora.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Attention ({mode}) LoRA r=8', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Attention ({mode}) LoRA r=8', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Attention-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # MHA with LoRA — both modes
    model_key = f'MHA (Heads={NUM_HEADS} | LoRA r=64)'
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_adv_single.loc[(model_key, mode), patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'MHA-{NUM_HEADS} ({mode}) LoRA r=64', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'MHA-{NUM_HEADS} ({mode}) LoRA r=64', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'MHA-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Hierarchical MHA with LoRA — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_hierarchical.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Hierarchical ({mode}) LoRA r=8', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Hierarchical ({mode}) LoRA r=8', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Hierarchical-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

df_nb05 = pd.DataFrame(rows)
df_nb05.to_csv("results_csv/nb05_results.csv", index=False)
print("Saved results_csv/nb05_results.csv")
display(df_nb05)

Saved results_csv/nb05_results.csv


,finetuning,pooling,patch_size,accuracy,macro_recall,macro_f1
0,05_LoRA,Mean,8,0.649230,0.385286,0.393846
1,05_LoRA,Attention-shared,8,0.637064,0.354620,0.355348
2,05_LoRA,Attention-indep,8,0.632603,0.369298,0.372289
3,05_LoRA,MHA-shared,8,0.618005,0.357222,0.357541
4,05_LoRA,MHA-indep,8,0.624088,0.386772,0.383386
5,05_LoRA,Hierarchical-shared,8,0.582320,0.309307,0.297260
6,05_LoRA,Hierarchical-indep,8,0.587591,0.314547,0.302811
7,05_LoRA,Mean,16,0.641525,NaN,NaN
8,05_LoRA,Attention-shared,16,0.636659,NaN,NaN
9,05_LoRA,Attention-indep,16,0.617194,NaN,NaN


In [ ]:
df_patch_8_metrics.to_csv("results_csv/nb05_patch_8_metrics.csv", index=True)